In [ ]:
'''Apply LIME (Local Interpretable Model-Agnostic Explanations) for explaining individual model predictions, 
generating interpretable surrogate models to enhance trust in black-box classifiers.'''

In [ ]:
# import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from lime.lime_tabular import LimeTabularExplainer

In [ ]:
# load dataset and create classification target
df = pd.read_csv('global_pharmacy_sales_data.csv')
df_sample = df.sample(n=5000, random_state=42).copy()
df_sample['high_sales'] = (df_sample['units_sold'] > df_sample['units_sold'].median()).astype(int)

for col in ['category', 'age_group', 'region']:
    df_sample[col] = LabelEncoder().fit_transform(df_sample[col])

features = ['unit_price', 'stock_level', 'expiry_days_remaining', 'covid_flag', 'category', 'age_group', 'region', 'month']
X = df_sample[features]
y = df_sample['high_sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# train black box random forest classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train.values, y_train.values)
print('test accuracy:', clf.score(X_test.values, y_test.values))

In [ ]:
# initialize lime tabular explainer
explainer = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=features,
    class_names=['normal_sales', 'high_sales'],
    mode='classification',
    random_state=42
)

In [ ]:
# explain first individual prediction using lime surrogate model
sample_idx = 0
true_class = y_test.iloc[sample_idx]
pred_prob = clf.predict_proba(X_test.values[sample_idx:sample_idx+1])[0]
print(f'actual: {true_class}, predicted probability: {pred_prob}')

exp_1 = explainer.explain_instance(
    data_row=X_test.values[sample_idx],
    predict_fn=clf.predict_proba,
    num_features=6
)
fig = exp_1.as_pyplot_figure()
plt.title('lime surrogate explanation for instance 1')
plt.tight_layout()
plt.show()

In [ ]:
# explain second individual prediction using lime surrogate model
sample_idx_2 = 1
true_class_2 = y_test.iloc[sample_idx_2]
pred_prob_2 = clf.predict_proba(X_test.values[sample_idx_2:sample_idx_2+1])[0]
print(f'actual: {true_class_2}, predicted probability: {pred_prob_2}')

exp_2 = explainer.explain_instance(
    data_row=X_test.values[sample_idx_2],
    predict_fn=clf.predict_proba,
    num_features=6
)
fig = exp_2.as_pyplot_figure()
plt.title('lime surrogate explanation for instance 2')
plt.tight_layout()
plt.show()

In [ ]:
# display surrogate rule weights for local interpretability
weights = pd.DataFrame(exp_1.as_list(), columns=['feature_condition', 'weight'])
plt.figure(figsize=(8, 4))
plt.barh(weights['feature_condition'], weights['weight'], color='cornflowerblue')
plt.title('interpretable surrogate local feature contributions')
plt.tight_layout()
plt.show()